In [125]:
import pandas as pd
import numpy as np
import os
from tqdm import tqdm
import sys; sys.path.append("/data/jerrylee/pjt/BIGFAM.v.0.1")
import BIGFAM.tools as tools

# Step 1. UKB

## Step 1.1 relative information

In [116]:
# parameters
info_fn = "/data/jerrylee/pjt/BIGFAM.v.0.1/data/UKB/relative_information/relatives.raw.info"

In [117]:
info_colns = ["DOR", "relationship",
              "volid", "relid", 
              "volage", "relage", 
              "volsex", "relsex", 
              "Erx"]

In [118]:
# relative information format
df_pair = pd.read_csv(info_fn, sep='\t')

# convert to new column name
df_pair = df_pair.rename(columns={
    "eid": "volid", 
    "eid_rel": "relid",
    "age": "volage",
    "age_rel": "relage",
    "sex": "volsex",
    "sex_rel": "relsex",
    "rel_type": "relationship",
    # "rx": "Erx",
    # "Erx": "rx"
})

# recode sex
df_pair["volsex"] = df_pair["volsex"].replace({0: "F", 1: "M"})
df_pair["relsex"] = df_pair["relsex"].replace({0: "F", 1: "M"})

df_pair[info_colns].head()

,DOR,relationship,volid,relid,volage,relage,volsex,relsex,Erx
0,1,DS,1000094,3653174,65,64,F,F,0.75
1,1,NaN,1000220,1691267,64,64,F,F,NaN
2,1,NaN,1000286,1571411,53,70,F,F,NaN
3,1,NaN,1000295,1045127,60,41,F,F,NaN
4,1,NaN,1000476,3599303,50,51,F,M,NaN


In [137]:
recode_rel = {
    # dor1
    'SB': 'son-brother',
    'SS_DB': 'different-sex-sibling', 
    'DS': 'daughter-sister', 
     
    'FS': 'son-father',
    'MS': 'son-mother',
    'FD': 'daughter-father', 
    'MD': 'daughter-mother', 
    
    # dor2
    "DMS": "daughter-mother-sister",
    "DFB": "daughter-father-brother",
    "SMS": "son-mother-sister",
    "SFB": "son-father-brother",
    "DFS": "daughter-father-sister",
    "DMB": "daughter-mother-brother",
    "SMB": "son-mother-brother",
    
    # dor3
    "DMSD": "daughter-mother-sister-daughter",
    "DFSD": "daughter-father-sister-daughter",
    "SMSS": "son-mother-sister-son",
    "SMSD_DMSS": "son-mother-sister-daughter",
    "SMBD_DFSS": "son-mother-brother-daughter",
    "DFBD_DMBD": "daughter-(father/mother)-brother-daughter",
    "SFBS_SFSS_SMBS": "son-(father-sister)/(father-sister)/(mother-brother)-son",
    "SFBD_SFSD_DFBS_DMBS": "son-(father-brother)/(father-sister)-daughter",
}

recode_rcode = {
    # dor1
    'SB': 'SB',
    'SS_DB': 'SB',
    'DS': 'SB', 
     
    'FS': 'PC',
    'MS': 'PC',
    'FD': 'PC',
    'MD': 'PC',
    
    # dor2
    "DMS": "AV",
    "DFB": "AV",
    "SMS": "AV",
    "SFB": "AV",
    "DFS": "AV",
    "DMB": "AV",
    "SMB": "AV",
    
    # dor3
    "DMSD": "1C",
    "DFSD": "1C",
    "SMSS": "1C",
    "SMSD_DMSS": "1C",
    "SMBD_DFSS": "1C",
    "DFBD_DMBD": "1C",
    "SFBS_SFSS_SMBS": "1C",
    "SFBD_SFSD_DFBS_DMBS": "1C",
}

In [144]:
df_new = df_pair.copy()
df_new["rcode"] = df_new["relationship"]
df_new = df_new.replace(
    to_replace={
        "relationship": recode_rel,
        "rcode": recode_rcode
    })
df_new

,DOR,volid,volage,volsex,relid,relage,relsex,sex_type,relationship,Era,ra,rx,Erx,rcode
0,1,1000094,65,F,3653174,64,F,ff,daughter-sister,0.500,0.4942,0.632092,0.75,SB
1,1,1000220,64,F,1691267,64,F,ff,NaN,0.500,0.5520,0.927974,NaN,NaN
2,1,1000286,53,F,1571411,70,F,ff,NaN,0.500,0.4936,NaN,NaN,NaN
3,1,1000295,60,F,1045127,41,F,ff,NaN,0.500,0.4924,NaN,NaN,NaN
4,1,1000476,50,F,3599303,51,M,mf,NaN,0.500,0.4898,0.111175,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
81321,3,6023723,62,F,4863061,64,M,mf,son-(father-brother)/(father-sister)-daughter,0.125,0.1392,-0.000470,0.00,1C
81322,3,6024211,53,M,1209127,60,F,mf,NaN,0.125,0.1302,-0.032276,NaN,NaN
81323,3,6024384,62,M,1854265,44,M,mm,NaN,0.125,0.1162,0.103215,NaN,NaN
81324,3,6024486,58,M,3148753,56,M,mm,son-(father-sister)/(father-sister)/(mother-br...,0.125,0.1324,-0.040457,0.00,1C


In [145]:
df_new[["DOR", "rcode", "relationship", "volid", "relid", "volage", "relage", "volsex", "relsex", "Erx"]]

,DOR,rcode,relationship,volid,relid,volage,relage,volsex,relsex,Erx
0,1,SB,daughter-sister,1000094,3653174,65,64,F,F,0.75
1,1,NaN,NaN,1000220,1691267,64,64,F,F,NaN
2,1,NaN,NaN,1000286,1571411,53,70,F,F,NaN
3,1,NaN,NaN,1000295,1045127,60,41,F,F,NaN
4,1,NaN,NaN,1000476,3599303,50,51,F,M,NaN
...,...,...,...,...,...,...,...,...,...,...
81321,3,1C,son-(father-brother)/(father-sister)-daughter,6023723,4863061,62,64,F,M,0.00
81322,3,NaN,NaN,6024211,1209127,53,60,M,F,NaN
81323,3,NaN,NaN,6024384,1854265,62,44,M,M,NaN
81324,3,1C,son-(father-sister)/(father-sister)/(mother-br...,6024486,3148753,58,56,M,M,0.00


In [146]:
(df_new[["DOR", "rcode", "relationship", "volid", "relid", "volage", "relage", "volsex", "relsex", "Erx"]]
 .to_csv(
    "/data/jerrylee/pjt/BIGFAM.v.0.1/data/UKB/relative_information/relatives.formatted.info",
    sep='\t',
    index=False
 )
)

In [147]:
df_pair.dropna().groupby(["DOR", "relationship"]).size()

DOR  relationship       
1    DS                     3137
     FD                      949
     FS                      637
     MD                     1874
     MS                     1281
     SB                      665
     SS_DB                  1861
2    DFB                     286
     DFS                     330
     DMB                     229
     DMS                     579
     SFB                     586
     SMB                      98
     SMS                     346
3    DFBD_DMBD               921
     DFSD                    935
     DMSD                   1274
     SFBD_SFSD_DFBS_DMBS    5970
     SFBS_SFSS_SMBS         3369
     SMBD_DFSS               947
     SMSD_DMSS              1051
     SMSS                    287
dtype: int64

## Step 1.2. phenotype

In [2]:
pheno_path = "/data/jerrylee/pjt/BIGFAM.v.0.1/data/UKB/phenotype"

In [5]:
info_colns = ["eid", "pheno"]

In [ ]:
pheno_fns = os.listdir(pheno_path)
len(pheno_fns)

In [7]:
for pheno_fn in pheno_fns:
    pheno = pheno_fn.split(".")[0]
    fn = f"{pheno_path}/{pheno_fn}"
    
    df_pheno = (pd.read_csv(fn, sep='\t')
                .rename(columns={"residual": "pheno"}))
    
    df_pheno[info_colns].to_csv(
        f"/data/jerrylee/pjt/BIGFAM.v.0.1/data/UKB/phenotype/{pheno}.phen",
        sep='\t',
        index=False
    )

# Step 2. GS

## Step 2.1 relative information

In [91]:
# parameters
info_fn = "/data/jerrylee/pjt/BIGFAM.v.0.1/data/GS/relative_information/relatives.raw.info"

In [92]:
# relative information format
info_colns = ["DOR", "rcode", "relationship",
              "volid", "relid", 
              "volage", "relage", 
              "volsex", "relsex", 
              "Erx"]
df_pair = pd.read_csv(info_fn, sep='\t')

# convert to new column name
df_pair = df_pair.rename(columns={
    # "eid": "volid", 
    # "eid_rel": "relid",
    # "age": "volage",
    # "age_rel": "relage",
    # "sex": "volsex",
    # "sex_rel": "relsex",
    "rel_type": "relationship",
})

# # recode sex
# df_pair["volsex"] = df_pair["volsex"].replace({0: "F", 1: "M"})
# df_pair["relsex"] = df_pair["relsex"].replace({0: "F", 1: "M"})

df = df_pair[info_colns].copy()
df

,DOR,rcode,relationship,volid,relid,volage,relage,volsex,relsex,Erx
0,1,SB,DS,18826,21244,50,36,F,F,0.750000
1,1,SB,DS,21244,18826,36,50,F,F,0.750000
2,1,SB,SS+DB,34422,23884,33,35,F,M,0.353553
3,1,SB,SS+DB,23884,34422,35,33,M,F,0.353553
4,1,PC,DM,79198,67531,66,44,F,F,0.500000
...,...,...,...,...,...,...,...,...,...,...
57589,2,AV,DM_SS+DB,80945,18001,30,66,F,M,0.176777
57590,1,PC,DM,79563,80945,60,30,F,F,0.500000
57591,2,AV,SS+DB_DM,18001,80945,66,30,M,F,0.176777
57592,1,SB,SS+DB,18001,79563,66,60,M,F,0.353553


### Step 2.1.1 check strange info

In [93]:
df_kinship = pd.read_csv("/data/jerrylee/data/GS/kinpair.xls", sep='\t')
df_pedigree = pd.read_csv("/data/jerrylee/data/GS/pedigree.xls", sep='\t',
                          dtype={"volid": int, "father": int, "mother": int})
df_pheno = pd.read_csv("/data/jerrylee/data/GS/phenotypes.xls", sep='\t')

In [94]:
idx_to_remove = []

In [95]:
is_dor1 = (df["DOR"] == 1)

for rel in (df.loc[is_dor1, ["relationship"]]
        .apply(lambda x: ":".join(map(str, x)), axis=1)
        .unique()):
    if len(rel.split("_")) != 1:
        tmp = df[is_dor1 & (df["relationship"] == rel)]
        print(tmp)
        idx_to_remove = idx_to_remove + list(tmp.index)
        

In [96]:
# DOR2
is_dor2 = (df["DOR"] == 2)

for rel in (df.loc[is_dor2, ["relationship"]]
        .apply(lambda x: ":".join(map(str, x)), axis=1)
        .unique()):
    if len(rel.split("_")) != 2:
        tmp = df[is_dor2 & (df["relationship"] == rel)]
        print(tmp)
        idx_to_remove = idx_to_remove + list(tmp.index)

In [97]:
# DOR3
is_dor3 = (df["DOR"] == 3)

for rel in (df.loc[is_dor3, ["relationship"]]
        .apply(lambda x: ":".join(map(str, x)), axis=1)
        .unique()):
    if len(rel.split("_")) != 3:
        tmp = df[is_dor3 & (df["relationship"] == rel)]
        print(tmp)
        idx_to_remove = idx_to_remove + list(tmp.index)

In [98]:
# df_kinship[(df_kinship["volid"] == 59092) & (df_kinship["relid"] == 98110)]
# df_pedigree[df_pedigree["famid"] == 144]

# # annotated as 1C, actually couple

In [99]:
# df_kinship[(df_kinship["volid"] == 91547) & (df_kinship["relid"] == 4483)]
# df_pedigree[df_pedigree["famid"] == 5392]

# # annotated as 1C, actually couple

In [100]:
df_filtered = (df.loc[~df.index.isin(idx_to_remove)]
               .reset_index(drop=True))
df_filtered

,DOR,rcode,relationship,volid,relid,volage,relage,volsex,relsex,Erx
0,1,SB,DS,18826,21244,50,36,F,F,0.750000
1,1,SB,DS,21244,18826,36,50,F,F,0.750000
2,1,SB,SS+DB,34422,23884,33,35,F,M,0.353553
3,1,SB,SS+DB,23884,34422,35,33,M,F,0.353553
4,1,PC,DM,79198,67531,66,44,F,F,0.500000
...,...,...,...,...,...,...,...,...,...,...
57589,2,AV,DM_SS+DB,80945,18001,30,66,F,M,0.176777
57590,1,PC,DM,79563,80945,60,30,F,F,0.500000
57591,2,AV,SS+DB_DM,18001,80945,66,30,M,F,0.176777
57592,1,SB,SS+DB,18001,79563,66,60,M,F,0.353553


### Step 2.1.2 set `relationship` unique

"SS_DM" = "DM_SS"

DOR1

In [101]:
def swap_order(text):
    splitted = text.split("_")
    if len(splitted) == 2:
        return f"{splitted[1]}_{splitted[0]}"
    if len(splitted) == 3:
        return f"{splitted[2]}_{splitted[1]}_{splitted[0]}"

In [102]:
pcsb_recode = {
    'SB': 'son-brother',
    'SS+DB': 'different-sex-sibling', 
    'DS': 'daughter-sister', 
     
    'SF': 'son-father',
    'SM': 'son-mother',
    'DF': 'daughter-father', 
    'DM': 'daughter-mother', 
}

In [103]:
df_new = pd.DataFrame()
df_to_concat = tools.remove_duplicate_relpair(
    df_filtered[df_filtered["DOR"] == 1].copy(),
    ["volid", "relid"]
)

df_to_concat = df_to_concat.replace(to_replace=
                                    {"relationship": pcsb_recode})
df_new = (pd.concat([df_new, df_to_concat], axis=0)
          .reset_index(drop=True))

len(df_new)

18258

DOR2

In [104]:
av_recode = {
    'SF_SB': 'son-father-brother',
    'SB_SF': 'son-father-brother',
    
    'SF_SS+DB': 'son-father-sister',
    'SS+DB_SF': 'son-father-sister',
    
    'SM_SS+DB': 'son-mother-brother',
    'SS+DB_SM': 'son-mother-brother',
    
    'SM_DS': 'son-mother-sister',
    'DS_SM': 'son-mother-sister',
    
    'DF_SB': 'daughter-father-brother',
    'SB_DF': 'daughter-father-brother',
    
    'DF_SS+DB': 'daughter-father-sister',
    'SS+DB_DF': 'daughter-father-sister',
    
    'DM_SS+DB': 'daughter-mother-brother',
    'SS+DB_DM': 'daughter-mother-brother',
    
    'DS_DM': 'daughter-mother-sister',
    'DM_DS': 'daughter-mother-sister',
}

gp_recode = {
    'SF_SF': 'son-father-father',
    
    'SF_SM': 'son-father-mother',
    'SM_SF': 'son-father-mother',
    
    'DM_SM': 'son-mother-mother',
    'SM_DM': 'son-mother-mother',
    
    'SF_DF': 'daughter-father-father',
    'DF_SF': 'daughter-father-father',
    
    # 
    
    'DM_DF': 'daughter-mother-father',
    'DF_DM': 'daughter-mother-father',
    
    'DM_DM': 'daughter-mother-mother',
    
    # 'DF_SM',
    # 'SM_DF'
}

hsb_recode = {
    'SF_SF': 'son-father-son', 
    'SF_DF': 'son-father-daughter', 
    'SM_SM': 'son-mother-son', 
    'SM_DM': 'son-mother-daughter', 
    
    'DF_SF': 'daughter-father-son', 
    'DF_DF': 'daughter-father-daughter',
    'DM_SM': 'daughter-mother-son',
    'DM_DM': 'daughter-mother-daughter', 
}

In [105]:
is_dor2 = (df_filtered["DOR"] == 2)

df_rrelerx = pd.DataFrame(columns=["rcode", "relationship", "Erx"])
df_dor2 = df_filtered[is_dor2].copy()
stop=False

for rcode in df_dor2["rcode"].unique():
    tmp = df_dor2[df_dor2["rcode"] == rcode]
    for rel in tmp["relationship"].unique():
        erx = tmp.loc[tmp["relationship"] == rel, "Erx"].unique()
        
        if len(erx) != 1:
            print(rcode, rel, erx)
        else:
            df_rrelerx.loc[len(df_rrelerx)] = [rcode, rel, erx[0]]
            
for _, row in (df_rrelerx.groupby(["rcode", "Erx"]).size()
               .to_frame().reset_index().iterrows()):
    rcode, erx, _ = row.values
    
    tmp = df_rrelerx[(df_rrelerx["rcode"] == rcode) 
           & (((df_rrelerx["Erx"] - erx) < 1e-4)
           & ((df_rrelerx["Erx"] - erx) > -1e-4))]
    
    for rel in tmp["relationship"].unique():
        ttmp = df_dor2[(df_dor2["rcode"] == rcode) & (df_dor2["relationship"] == rel)]
        ttmp_flip = df_dor2[(df_dor2["rcode"] == rcode) & (df_dor2["relationship"] == swap_order(rel))]
        
        volsex = ttmp["volsex"].unique()
        relsex = ttmp["relsex"].unique()
        volsex_flip = ttmp_flip["volsex"].unique()
        relsex_flip = ttmp_flip["relsex"].unique()
        
        # is rel_flip == rel
        if (len(volsex) == 1) | (len(relsex) == 1):
            if ((volsex[0] == relsex_flip[0]) & (relsex[0] == volsex_flip[0])):
                if rcode == "AV":
                    tttmp = ttmp.replace(to_replace={
                        'relationship': {rel:av_recode[rel]}
                    })
                elif rcode == "GP":
                    tttmp = ttmp.replace(to_replace={
                        'relationship': {rel:gp_recode[rel]}
                    })
                    # print("==========")
                    # print(rcode, rel)
                    # print(ttmp)
                    # print()
                    # print(tttmp)
                    
                elif rcode == "HSB":
                    tttmp = ttmp.replace(to_replace={
                        'relationship': {rel:hsb_recode[rel]}
                    })
                    # if (hsb_recode[rel] == "daughter-father-son"):
                    #     print("==========")
                    #     print(rcode, rel)
                    #     print(ttmp)
                    #     print()
                    #     print(tttmp)
                else:
                    raise  
                
                tttmp = tools.remove_duplicate_relpair(
                        tttmp.copy(),
                        ["volid", "relid"]
                    )
                df_new = (pd.concat([df_new, tttmp], axis=0)
                            .reset_index(drop=True))
                      
            else:
                print("flip error", rcode, rel)
        # multiple relationship
        else:
            print("multiple relationship | ", rcode, rel)
            if (rel == "DF_SM") | (rel == "SM_DF"):
                ttmp_male = ttmp[ttmp["volsex"] == "M"]
                ttmp_female = ttmp[ttmp["volsex"] == "F"]
                
                ttmp_male = ttmp_male.replace(to_replace={
                        'relationship': {rel:'son-mother-father'}
                    })
                ttmp_male = tools.remove_duplicate_relpair(
                    ttmp_male.copy(),
                    ["volid", "relid"]
                )
                df_new = (pd.concat([df_new, ttmp_male], axis=0)
                            .reset_index(drop=True))
                
                ttmp_female = ttmp_female.replace(to_replace={
                        'relationship': {rel:'daughter-father-mother'}
                    })
                ttmp_female = tools.remove_duplicate_relpair(
                    ttmp_female.copy(),
                    ["volid", "relid"]
                )
                df_new = (pd.concat([df_new, ttmp_female], axis=0)
                            .reset_index(drop=True))
            else:
                raise
        
    #         stop=True
    #         break
    # if stop:
    #     break
            
            
            

multiple relationship |  GP DF_SM
multiple relationship |  GP SM_DF


dor3

In [106]:
c_recode = {
    'SF_SB_SF': 'son-father-brother-son',
    
    'SF_SB_DF': 'son-father-brother-daughter',
    'DF_SB_SF': 'son-father-brother-daughter',
    
    'SF_SS+DB_SM': 'son-father-sister-son',
    'SM_SS+DB_SF': 'son-father-sister-son',
    
    'SF_SS+DB_DM': 'son-father-sister-daughter',
    'DM_SS+DB_SF': 'son-father-sister-daughter',
    
    'SM_SS+DB_DF': 'son-mother-brother-daughter',
    'DF_SS+DB_SM': 'son-mother-brother-daughter',
    'SM_DS_SM': 'son-mother-sister-son',
    
    'SM_DS_DM': 'son-mother-sister-daughter',
    'DM_DS_SM': 'son-mother-sister-daughter',
    
    'DF_SB_DF': 'daughter-father-brother-daughter',
    'DM_SS+DB_DF': 'daughter-father-sister-daughter',
    'DF_SS+DB_DM': 'daughter-father-sister-daughter',
    
    'DM_DS_DM': 'daughter-mother-sister-daughter',
    
}

gg_recode = {
    'SF_SF_SM': 'son-father-father-mother', 
    'SM_SF_SF': 'son-father-father-mother', 
}


hav_recode = {
    'SF_SF_SF': 'son-father-father-son',
    
    'SF_SF_DF': 'son-father-father-daughter',
    'DF_SF_SF': 'son-father-father-daughter',
    
    'SM_SM_SF': 'son-father-mother-son',
    'SF_SM_SM': 'son-father-mother-son',
    
    'SF_SM_DM': 'son-father-mother-daughter',
    'DM_SM_SF': 'son-father-mother-daughter',
    
    'SM_DF_DF': 'son-mother-father-daughter',
    'DF_DF_SM': 'son-mother-father-daughter',
    
    'SM_DM_SM': 'son-mother-mother-son',
    
    'SM_DM_DM': 'son-mother-mother-daughter',
    'DM_DM_SM': 'son-mother-mother-daughter',
    
    
    'DF_SF_DF': 'daughter-father-father-daughter',
    
    'SM_SM_DF': 'daughter-father-mother-son',
    'DF_SM_SM': 'daughter-father-mother-son',
    
    'DM_SM_DF': 'daughter-father-mother-daughter',
    'DF_SM_DM': 'daughter-father-mother-daughter',
    
    'DF_DF_DM': 'daughter-mother-father-daughter',
    'DM_DF_DF': 'daughter-mother-father-daughter',
    
    'DM_DM_DM': 'daughter-mother-mother-daughter',
}

In [107]:
is_dor3 = (df_filtered["DOR"] == 3)

df_rrelerx = pd.DataFrame(columns=["rcode", "relationship", "Erx"])
df_dor3 = df_filtered[is_dor3].copy()
stop=False

for rcode in df_dor3["rcode"].unique():
    tmp = df_dor3[df_dor3["rcode"] == rcode]
    for rel in tmp["relationship"].unique():
        erx = tmp.loc[tmp["relationship"] == rel, "Erx"].unique()
        
        if len(erx) != 1:
            print(rcode, rel, erx)
        else:
            df_rrelerx.loc[len(df_rrelerx)] = [rcode, rel, erx[0]]
            
for _, row in (df_rrelerx.groupby(["rcode", "Erx"]).size()
               .to_frame().reset_index().iterrows()):
    rcode, erx, _ = row.values
    
    tmp = df_rrelerx[(df_rrelerx["rcode"] == rcode) 
           & (((df_rrelerx["Erx"] - erx) < 1e-4)
           & ((df_rrelerx["Erx"] - erx) > -1e-4))]
    
    for rel in tmp["relationship"].unique():
        ttmp = df_dor3[(df_dor3["rcode"] == rcode) & (df_dor3["relationship"] == rel)]
        ttmp_flip = df_dor3[(df_dor3["rcode"] == rcode) & (df_dor3["relationship"] == swap_order(rel))]
        
        # ttmp = df_dor3[df_dor3["relationship"] == rel]
        # ttmp_flip = df_dor3[df_dor3["relationship"] == swap_order(rel)]
        
        volsex = ttmp["volsex"].unique()
        relsex = ttmp["relsex"].unique()
        volsex_flip = ttmp_flip["volsex"].unique()
        relsex_flip = ttmp_flip["relsex"].unique()
        
        # is rel_flip == rel
        if (len(volsex) == 1) | (len(relsex) == 1):
            if ((volsex[0] == relsex_flip[0]) & (relsex[0] == volsex_flip[0])):
                if rcode == "1C":
                    ttmp = ttmp.replace(to_replace={
                        'relationship': {rel:c_recode[rel]}
                    })
                elif rcode == "GG":
                    ttmp = ttmp.replace(to_replace={
                        'relationship': {rel:gg_recode[rel]}
                    })
                elif rcode == "HAV":
                    ttmp = ttmp.replace(to_replace={
                        'relationship': {rel:hav_recode[rel]}
                    })
                else:
                    raise  
                
                ttmp = tools.remove_duplicate_relpair(
                        ttmp.copy(),
                        ["volid", "relid"]
                    )
                df_new = (pd.concat([df_new, ttmp], axis=0)
                            .reset_index(drop=True))
                
            else:
                print("flip error", rcode, rel)
        # multiple relationship
        else:
            print("multiple relationship | ", rcode, rel)
    #         stop=True
    #         break
    # if stop:
    #     break
            
            
            

In [108]:
df_new

,DOR,rcode,relationship,volid,relid,volage,relage,volsex,relsex,Erx
0,1,SB,daughter-sister,18826,21244,50,36,F,F,0.750000
1,1,SB,different-sex-sibling,34422,23884,33,35,F,M,0.353553
2,1,PC,daughter-mother,79198,67531,66,44,F,F,0.500000
3,1,SB,daughter-sister,20399,67531,38,44,F,F,0.750000
4,1,SB,daughter-sister,67267,67531,43,44,F,F,0.750000
...,...,...,...,...,...,...,...,...,...,...
38001,3,HAV,son-mother-father-daughter,34570,78069,50,39,F,M,0.353553
38002,3,HAV,son-mother-father-daughter,97449,79361,40,29,F,M,0.353553
38003,3,HAV,son-mother-father-daughter,97449,5360,40,22,F,M,0.353553
38004,3,HAV,son-mother-father-daughter,15442,83545,35,29,F,M,0.353553


In [109]:
# n-pair of each relationship
df_new.groupby(["DOR", "relationship"]).size()

DOR  relationship                    
1    daughter-father                     2147
     daughter-mother                     3506
     daughter-sister                     3180
     different-sex-sibling               3797
     son-brother                         1470
     son-father                          1716
     son-mother                          2442
2    daughter-father-brother             1052
     daughter-father-daughter              14
     daughter-father-father                86
     daughter-father-mother               196
     daughter-father-sister              1440
     daughter-father-son                   11
     daughter-mother-brother             1716
     daughter-mother-daughter             162
     daughter-mother-father               200
     daughter-mother-mother               229
     daughter-mother-sister              3206
     daughter-mother-son                  149
     son-father-brother                   978
     son-father-daughter                  

In [110]:
df_new

,DOR,rcode,relationship,volid,relid,volage,relage,volsex,relsex,Erx
0,1,SB,daughter-sister,18826,21244,50,36,F,F,0.750000
1,1,SB,different-sex-sibling,34422,23884,33,35,F,M,0.353553
2,1,PC,daughter-mother,79198,67531,66,44,F,F,0.500000
3,1,SB,daughter-sister,20399,67531,38,44,F,F,0.750000
4,1,SB,daughter-sister,67267,67531,43,44,F,F,0.750000
...,...,...,...,...,...,...,...,...,...,...
38001,3,HAV,son-mother-father-daughter,34570,78069,50,39,F,M,0.353553
38002,3,HAV,son-mother-father-daughter,97449,79361,40,29,F,M,0.353553
38003,3,HAV,son-mother-father-daughter,97449,5360,40,22,F,M,0.353553
38004,3,HAV,son-mother-father-daughter,15442,83545,35,29,F,M,0.353553


In [113]:
df_new[df_new["volid"] == 18826]

,DOR,rcode,relationship,volid,relid,volage,relage,volsex,relsex,Erx
0,1,SB,daughter-sister,18826,21244,50,36,F,F,0.75


In [111]:
n_indiv = len(set(df_new["volid"]) | set(df_new["relid"]))
n_pair = len(df_new)

n_pair, n_indiv

(38006, 18006)

In [112]:
df_new.groupby("DOR").size()

DOR
1    18258
2    15114
3     4634
dtype: int64

In [81]:
df_new.to_csv(
    "/data/jerrylee/pjt/BIGFAM.v.0.1/data/GS/relative_information/relatives.formatted.info",
    sep='\t',
    index=False
)

## Step 2.2 phenotype

In [10]:
pheno_path = "/data/jerrylee/pjt/BIGFAM.v.0.1/data/GS/phenotype"

In [11]:
info_colns = ["eid", "pheno"]

In [12]:
pheno_fns = os.listdir(pheno_path)
len(pheno_fns)

42

In [15]:
for pheno_fn in tqdm(pheno_fns):
    pheno = pheno_fn.split(".")[0]
    fn = f"{pheno_path}/{pheno_fn}"
    
    df_pheno = (pd.read_csv(fn, sep='\t')
                .rename(columns={"residual": "pheno"}))
    
    df_pheno[info_colns].to_csv(
        f"/data/jerrylee/pjt/BIGFAM.v.0.1/data/GS/phenotype/{pheno}.phen",
        sep='\t',
        index=False
    )

100%|██████████| 42/42 [00:03<00:00, 12.52it/s]
